In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import numpy as np
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # GPUの演算（CuDNN）を確定的にする設定
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
# 1. モデルとトークナイザーの準備
model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_sae, l1_coeff):
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        self.l1_coeff = l1_coeff

        # Encoder: 入力を高次元に投影し、ReLUで活性化
        self.encoder = nn.Linear(d_in, d_sae)
        # 最初は入力信号のみに反応し、その後の学習（L1正則化）によって、不要なユニットが自然に負のバイアスを持ってオフになっていく
        self.encoder.bias.data.zero_()

        # Decoder: 高次元から元の次元へ復元
        self.decoder = nn.Linear(d_sae, d_in, bias=False)
        # 初期状態でデコーダの重みのノルムを1に設定
        self.set_decoder_norm_to_unit()

        # 入力の平均を引くためのバイアス（オプションだが精度向上のため推奨）
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def set_decoder_norm_to_unit(self):
        """デコーダの各列（各特徴量ベクトル）のL2ノルムを1に固定する"""
        with torch.no_grad():
            # dim=0（行方向）で計算し、各列の長さを1にする
            norm = torch.norm(self.decoder.weight, dim=0, keepdim=True)
            self.decoder.weight /= norm

    def forward(self, x):
        # 1. 中心化（デコーダバイアスを引く）
        x_centered = x - self.b_dec

        # 2. エンコード（スパースな特徴量 f の抽出）
        # f = ReLU(W_enc * x + b_enc)
        f = F.relu(self.encoder(x_centered))

        # 3. デコード（再構成）
        # x_hat = W_dec * f + b_dec
        x_reconstruct = self.decoder(f) + self.b_dec

        # 4. 損失の計算
        # 再構成誤差 (MSE)
        reconstruction_loss = F.mse_loss(x_reconstruct, x)
        # スパース性ペナルティ (L1)
        l1_loss = self.l1_coeff * f.sum(dim=-1).mean()

        loss = reconstruction_loss + l1_loss

        return x_reconstruct, loss, f

In [5]:
# 1. データセットの準備 (日本語 Wikipedia)
dataset = load_dataset(
    "wikimedia/wikipedia", 
    "20231101.ja", 
    split="train",
    streaming=True # メモリ節約のためストリーミングを推奨
)

def get_activations_loader(model, tokenizer, dataset, batch_size=32, max_tokens=100000):
    """
    LLMから活性化値を抽出し、シャッフルされたデータローダーとして返す
    """
    model.eval()
    all_acts = []
    token_count = 0
    
    # フックでデータを貯めるリスト
    local_storage = []
    def hook_fn(module, input, output):
        # (batch, seq, dim) -> (batch * seq, dim)
        local_storage.append(output.view(-1, output.shape[-1]).detach().cpu())

    handle = model.model.layers[10].mlp.register_forward_hook(hook_fn)

    print("Extracting activations from LLM...")
    for i, example in enumerate(dataset):
        inputs = tokenizer(example["text"], return_tensors="pt", truncation=True, max_length=512).to(model.device)
        
        with torch.no_grad():
            model(**inputs)
        
        # 溜まったデータをall_actsに移動
        if local_storage:
            acts = torch.cat(local_storage, dim=0)
            all_acts.append(acts)
            token_count += acts.shape[0]
            local_storage.clear()
            
        if token_count >= max_tokens:
            break
            
    handle.remove()
    
    # 全データを結合してシャッフル
    final_acts = torch.cat(all_acts, dim=0)
    # インデックスをランダムに入れ替える
    perm = torch.randperm(final_acts.shape[0])
    final_acts = final_acts[perm]
    
    return DataLoader(final_acts, batch_size=batch_size, shuffle=True)

# --- 学習の実行部分 ---

# 10万トークン分のデータを抽出
data_loader = get_activations_loader(model, tokenizer, dataset, batch_size=1024, max_tokens=100000)

# SAEの初期化
sae = SparseAutoencoder(d_in=2048, d_sae=32768, l1_coeff=0.001).to("cuda")
sae.to(torch.bfloat16)
optimizer = torch.optim.Adam(sae.parameters(), lr=5e-4)

print("Starting SAE Training...")
for epoch in range(32):
    total_loss = 0
    for batch in data_loader:
        batch = batch.to("cuda").to(torch.bfloat16)
        
        # 順伝播
        _, loss, f = sae(batch)
        
        # 逆伝播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 重要：デコーダの正規化
        sae.set_decoder_norm_to_unit()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch}: Loss {total_loss/len(data_loader):.6f}")

Extracting activations from LLM...
Starting SAE Training...
Epoch 0: Loss 1.019018
Epoch 1: Loss 0.605705
Epoch 2: Loss 0.539378
Epoch 3: Loss 0.507102
Epoch 4: Loss 0.488419
Epoch 5: Loss 0.475911
Epoch 6: Loss 0.466383
Epoch 7: Loss 0.458965
Epoch 8: Loss 0.451606
Epoch 9: Loss 0.445174
Epoch 10: Loss 0.440361
Epoch 11: Loss 0.436454
Epoch 12: Loss 0.433318
Epoch 13: Loss 0.430043
Epoch 14: Loss 0.426807
Epoch 15: Loss 0.423414
Epoch 16: Loss 0.419902
Epoch 17: Loss 0.416371
Epoch 18: Loss 0.412977
Epoch 19: Loss 0.409604
Epoch 20: Loss 0.406250
Epoch 21: Loss 0.403232
Epoch 22: Loss 0.400331
Epoch 23: Loss 0.397806
Epoch 24: Loss 0.395222
Epoch 25: Loss 0.392953
Epoch 26: Loss 0.390763
Epoch 27: Loss 0.388711
Epoch 28: Loss 0.386837
Epoch 29: Loss 0.385002
Epoch 30: Loss 0.383464
Epoch 31: Loss 0.381846


In [6]:
@torch.no_grad()
def collect_interpretation_data(model, tokenizer, dataset, n_samples=10000):
    """
    活性化値と、対応するトークンIDをセットで保存する
    """
    model.eval()
    all_acts = []
    all_tokens = []
    
    # フックの再定義
    local_acts = []
    def hook_fn(m, i, o):
        local_acts.append(o.view(-1, o.shape[-1]).detach().cpu())

    handle = model.model.layers[10].mlp.register_forward_hook(hook_fn)

    for i, example in enumerate(dataset):
        inputs = tokenizer(example["text"], return_tensors="pt", truncation=True, max_length=128).to(model.device)
        model(**inputs)
        
        # トークンIDも保存 (shape: [1, seq_len] -> [seq_len])
        all_tokens.append(inputs.input_ids[0].cpu())
        
        if local_acts:
            all_acts.append(torch.cat(local_acts, dim=0))
            local_acts.clear()
        
        if len(torch.cat(all_tokens)) >= n_samples:
            break

    handle.remove()
    return torch.cat(all_acts), torch.cat(all_tokens)

def get_top_activating_tokens(sae, activations, tokens, tokenizer, unit_id, top_k=10):
    sae.eval()
    with torch.no_grad():
        # SAEを通して特徴量 f を計算
        # activations は [N, 2048]
        # dtype を合わせるのを忘れずに！
        acts_input = activations.to("cuda").to(torch.bfloat16)
        f = torch.nn.functional.relu(sae.encoder(acts_input - sae.b_dec))
        
        # 特定ユニットの全データにおける値を取得
        unit_values = f[:, unit_id].cpu()
        
        # 値が大きい順にインデックスを取得
        top_values, top_indices = torch.topk(unit_values, k=top_k)
        
        print(f"--- Unit #{unit_id} Interpretation ---")
        for val, idx in zip(top_values, top_indices):
            if val > 0:
                token_text = tokenizer.decode([tokens[idx]])
                # 前後の文脈も出すとより分かりやすい（今回は単語のみ）
                print(f"Activation: {val:.4f} | Token: '{token_text}'")

In [7]:
# 1. データの収集 (1万トークン分)
acts, tokens = collect_interpretation_data(model, tokenizer, dataset, n_samples=10000)

# 2. 特定のユニットを調べてみる
# 例えば、0番から順に見ていくのも手ですが、まずは「よく働いている」ユニットを探します
# ここでは例として「ユニット #1024」を調査
get_top_activating_tokens(sae, acts, tokens, tokenizer, unit_id=1026)

--- Unit #1024 Interpretation ---
Activation: 0.0425 | Token: 'et'


In [12]:
get_top_activating_tokens(sae, acts, tokens, tokenizer, unit_id=1026)

--- Unit #1026 Interpretation ---
Activation: 3.1562 | Token: ' License'
Activation: 2.5000 | Token: '東'
Activation: 2.0000 | Token: '西'
Activation: 1.9219 | Token: '東'
Activation: 1.8984 | Token: '東'
Activation: 1.7266 | Token: 'マ'
Activation: 1.6406 | Token: '西'
Activation: 1.5859 | Token: '東'
Activation: 1.3125 | Token: '�'
Activation: 1.2578 | Token: '東'


In [8]:
@torch.no_grad()
def evaluate_sae(sae, activations):
    sae.eval()
    x = activations.to("cuda").to(torch.bfloat16)
    
    # 順伝播
    x_reconstruct, loss, f = sae(x)
    
    # 1. L0 Norm (スパース性)
    # 各トークンで活性化したユニット数の平均
    l0 = (f > 0).float().sum(dim=-1).mean().item()
    
    # 2. 再構成精度 (FVU)
    per_token_mse = (x_reconstruct - x).pow(2).sum(dim=-1)
    total_variance = (x - x.mean(dim=0)).pow(2).sum(dim=-1)
    fvu = (per_token_mse.sum() / total_variance.sum()).item()
    
    # 3. Dead Units
    # 全データを通して一度も活性化しなかったユニット
    is_dead = (f <= 0).all(dim=0)
    dead_unit_count = is_dead.sum().item()
    dead_unit_pct = (dead_unit_count / sae.d_sae) * 100

    print(f"\n--- SAE Evaluation Metrics ---")
    print(f"L0 (Active Units per Token): {l0:.2f} / {sae.d_sae}")
    print(f"FVU (Lower is better):       {fvu:.4f}")
    print(f"Explained Variance:          {1 - fvu:.4f}")
    print(f"Dead Units:                  {dead_unit_count} ({dead_unit_pct:.2f}%)")
    
    return {"l0": l0, "fvu": fvu, "dead_units": dead_unit_pct}

# 実行
metrics = evaluate_sae(sae, acts)


--- SAE Evaluation Metrics ---
L0 (Active Units per Token): 94.79 / 32768
FVU (Lower is better):       0.4277
Explained Variance:          0.5723
Dead Units:                  10772 (32.87%)


In [9]:
@torch.no_grad()
def find_interesting_units(sae, activations):
    sae.eval()
    acts_input = activations.to("cuda").to(torch.bfloat16)
    f = torch.nn.functional.relu(sae.encoder(acts_input - sae.b_dec))
    
    # 各ユニットが「0より大きい」値を出した回数をカウント
    activation_counts = (f > 0).sum(dim=0).cpu()
    
    # 頻度が 0.1% 〜 1% 程度のユニットが、特定の「概念」を持っていることが多い
    # (n_samples=10000 なら 10〜100回反応するユニット)
    target_units = torch.where((activation_counts > 10) & (activation_counts < 100))[0]
    return target_units.tolist()

# 候補となるユニットのリストを取得
candidate_units = find_interesting_units(sae, acts)
print(f"候補ユニット数: {len(candidate_units)}")

候補ユニット数: 349


In [22]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate_html_report(sae, activations, tokens, tokenizer, unit_ids, filename="sae_report.html"):
    sae.eval()
    
    # 1. すべてのデータに対する特徴量 f を一括計算 (メモリ節約のため小分けにするのもアリ)
    acts_input = activations.to("cuda").to(torch.bfloat16)
    # f shape: [N, d_sae]
    f = F.relu(sae.encoder(acts_input - sae.b_dec))
    
    html_content = ["""
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: sans-serif; background-color: #f4f4f9; padding: 20px; }
            .card { background: white; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 20px; padding: 15px; }
            .unit-title { color: #333; border-bottom: 2px solid #3498db; padding-bottom: 5px; }
            .token-box { display: inline-block; padding: 4px 8px; margin: 3px; border-radius: 4px; border: 1px solid #ddd; }
            .val-label { font-size: 0.8em; color: #666; display: block; }
            .token-text { font-weight: bold; font-size: 1.1em; }
        </style>
    </head>
    <body>
        <h1>SAE Feature Interpretation Report (Qwen3-1.7B)</h1>
    """]

    for uid in unit_ids:
        unit_values = f[:, uid].cpu()
        
        # 活性化値が 0 より大きいものだけを対象に、Top 20を取得
        top_k = min(20, (unit_values > 0).sum().item())
        if top_k == 0:
            continue
            
        top_values, top_indices = torch.topk(unit_values, k=top_k)
        
        html_content.append(f'<div class="card">')
        html_content.append(f'<h2 class="unit-title">Unit #{uid} (Frequency: {top_k} samples shown)</h2>')
        
        for val, idx in zip(top_values, top_indices):
            token_id = tokens[idx].item()
            token_text = tokenizer.decode([token_id])
            
            # 活性化値の強さに応じて背景の青色を濃くする (最大値を基準に正規化)
            opacity = min(1.0, val.item() / top_values[0].item())
            bg_color = f"rgba(52, 152, 219, {opacity * 0.6})" 
            
            html_content.append(f"""
                <div class="token-box" style="background-color: {bg_color};">
                    <span class="val-label">{val.item():.3f}</span>
                    <span class="token-text">'{token_text}'</span>
                </div>
            """)
        
        html_content.append('</div>')

    html_content.append("</body></html>")
    
    with open(filename, "w", encoding="utf-8") as f_out:
        f_out.write("\n".join(html_content))
    
    print(f"🎉 レポートが正常に生成されました: {filename}")

# --- 実行例 ---
# 頻度が中程度のユニットを20個ほど抽出してレポートにする
candidate_units = find_interesting_units(sae, acts) # 前のステップの関数
generate_html_report(sae, acts, tokens, tokenizer, candidate_units[:100])

🎉 レポートが正常に生成されました: sae_report.html


In [33]:
def compare_steering(model, tokenizer, sae, prompt, steer_unit_id, steer_strength=20.0, suppression_factor=0.0):
    """
    通常の状態と、特定のSAEユニットを操作した状態の出力を比較する
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # 生成のための共通設定
    gen_kwargs = {
        "max_new_tokens": 300,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "pad_token_id": tokenizer.eos_token_id,
    }

    # --- 1. 通常の状態での生成 ---
    # シード固定（比較を公平にするため）
    torch.manual_seed(42)
    with torch.no_grad():
        output_normal = model.generate(**inputs, **gen_kwargs)
    
    text_normal = tokenizer.decode(output_normal[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # --- 2. ステアリング介入状態での生成 ---
    def steering_hook(module, input, output):
        # 1. SAEで今の思考（特徴量 f）を抽出
        x_centered = output - sae.b_dec
        f = torch.nn.functional.relu(sae.encoder(x_centered))
        
        # 2. 全ユニットを抑制係数で掛け合わせる（0.0なら全消去）
        f_steered = f * suppression_factor
        
        # 3. ターゲットのユニットだけを高い値に固定
        f_steered[:, :, steer_unit_id] = steer_strength
        
        # 4. 元の空間に戻す
        return sae.decoder(f_steered) + sae.b_dec

    handle = model.model.layers[10].mlp.register_forward_hook(steering_hook)

    # 同じシードをセット（介入による差分だけを抽出するため）
    torch.manual_seed(42)
    with torch.no_grad():
        output_steered = model.generate(**inputs, **gen_kwargs)
    
    text_steered = tokenizer.decode(output_steered[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    handle.remove()

    # --- 結果の表示 ---
    print(f"\n{'='*20} 比較実験 {'='*20}")
    print(f"プロンプト: {prompt}")
    print(f"操作ユニット: #{steer_unit_id} ('神' 関連)")
    print(f"{'-'*49}")
    print(f"【通常の状態】\n{text_normal.strip()}")
    print(f"{'-'*49}")
    print(f"【ステアリング介入 (強度:{steer_strength})】\n{text_steered.strip()}")
    print(f"{'='*49}\n")

# --- 実行 ---
# 全く関係ない話題を振ってみる
compare_steering(model, tokenizer, sae, "今日の晩御飯の献立を考えてください。", steer_unit_id=3404, steer_strength=60.0, suppression_factor=0.0)


==================== 比較実験 ====================
プロンプト: 今日の晩御飯の献立を考えてください。
操作ユニット: #3404 ('神' 関連)
-------------------------------------------------
【通常の状態】
もちろん！今日の晩御飯の献立をいくつか提案します。ご希望の食材や調理法、ご飯の種類などがあれば、より具体的なメニューになります。

---

### 🍽️ 例：カジュアルな晩御飯（2人前）

#### 🥘 ファーストディナー（メインディナー）
- **味噌汁**（味噌、昆布、鯵や卵、人参、豆腐など）
- **焼き魚**（タタキ、ローストチキン、サバなど）
- **ご飯**（ご飯）

#### 🥗 サイドドishes
- **みかん**（フルーツ）
- **ごはん**（ごはん）
- **お味噌汁**（味噌、昆布、鯵など）

---

### 🍱 例：ヘルシーな晩御飯（2人前）

#### 🥘 ファーストディナー
- **豆腐のコンソメ煮**（豆腐、コンソメ、みりん、しいたけなど）
- **昆布味噌汁**（昆布、味噌、鯵、人参など）
- **ご飯**（ご飯）

#### 🥗 サイドドishes
- **ごはん**
- **みかん**
- **お豆腐のスープ
-------------------------------------------------
【ステアリング介入 (強度:60.0)】
もちろんです！以下は、今日の晩御飯の献立を考えてください。この献立は、食事をするための簡単な方法で、さまざまな食材と料理を含んでいます。

### 詳細な献立

1. **おにぎり**  
2. **おにぎり**（2つ）  
3. **おにぎり**（3つ）  
4. **おにぎり**（4つ）  
5. **おにぎり**（5つ）  
6. **おにぎり**（6つ）  
7. **おにぎり**（7つ）  
8. **おにぎり**（8つ）  
9. **おにぎり**（9つ）  
10. **おにぎり**（10つ）  
11. **おにぎり**（11つ）  
12. **おにぎり**（12つ）  
13. **おにぎり**（13つ）  
14. **おにぎり**（14つ）  
15